<a href="https://colab.research.google.com/github/gdiazh/lerobot_pi0_train_notebook/blob/main/training_smolvla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

Welcome to the **LeRobot SmolVLA training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `SmolVLA` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `SmolVLA` policy for 20,000 steps typically takes **about 5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer!

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg, and installs the package in editable mode with train, dataset and smolvla features.

In [6]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset, smolvla]"

Cloning into 'lerobot'...
remote: Enumerating objects: 55477, done.
remote: Counting objects: 100% (857/857), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 55477 (delta 722), reused 603 (delta 566), pack-reused 54620 (from 3)
Receiving objects: 100% (55477/55477), 250.00 MiB | 18.33 MiB/s, done.
Resolving deltas: 100% (35557/35557), done.
Filtering content: 100% (50/50), 69.11 MiB | 4.44 MiB/s, done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.5 M

## Weights & Biases login (optional)
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging. This step is optional, you can skip it. If you want to use W&B remember to change `--wandb.enable` to true in the next section.

In [ ]:
!wandb login

In [9]:
from google.colab import userdata
userdata.get('WANDB_API_KEY')
import os
import wandb
from google.colab import userdata

# Safely pull the key from your Colab Secrets
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')

# This will now authenticate automatically without prompting
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: gustavo-diaz (gustavo-diaz-tohoku-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## HF login

To upload your trained model to the hub you need to login with your Hugging Face account.
1. Run the cell below.
2. You will be asked to generate a token in the Hugging Face settings.
3. Select all checkboxes under Repositories when creating the token.
4. Paste the generated token into the command line below.

In [ ]:
!hf auth login

In [10]:
import os
from google.colab import userdata
from huggingface_hub import login

# Retrieve token and authenticate
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face!")
except Exception as e:
    print(f"Login failed. Ensure the 'HF_TOKEN' secret is enabled: {e}")

Successfully logged into Hugging Face!


In [14]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [41]:
mkdir -p /content/drive/colab/

In [38]:
ll -h /root/.config/Google/DriveFS/

total 256K
drwx------ 5 root 4.0K Jun 18 03:45 114353668654562025342/
-rw------- 1 root   11 Jun 18 01:49 cello_assert_history
-rw------- 1 root  64K Jun 18 01:49 experiments.db
-rw------- 1 root   20 Jun 18 01:49 first-run-info
-rw------- 1 root 1.4K Jun 18 01:49 global_feature_config
drwx------ 2 root 4.0K Jun 18 03:45 Logs/
-rw------- 1 root  12K Jun 18 01:49 metrics_store_sqlite.db
-rw------- 1 root  32K Jun 18 03:49 metrics_store_sqlite.db-shm
-rw------- 1 root  37K Jun 18 03:49 metrics_store_sqlite.db-wal
-rw------- 1 root    0 Jun 18 01:49 onboarding_data
-rw------- 1 root    4 Jun 18 01:49 pid.txt
-rw------- 1 root  113 Jun 18 01:49 preferences.json
-rw------- 1 root  36K Jun 18 01:49 root_preference_sqlite.db
-rw------- 1 root  32K Jun 18 01:49 root_preference_sqlite.db-shm
-rw------- 1 root    0 Jun 18 01:49 root_preference_sqlite.db-wal
-rw------- 1 root   24 Jun 18 01:49 ShellIpcPath
-rw------- 1 root  156 Jun 18 01:49 SyncTargets


In [12]:
mkdir -p /content/drive/colab

In [14]:
ls drive/

colab/


In [15]:
from google.colab import drive
drive.mount('/content/drive/colab/', force_remount=True)

Mounted at /content/drive/colab/


## Start training SmolVLA with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--batch_size=64`: means the model processes 64 training samples in parallel before doing one gradient update. Reduce this number if you have a GPU with low memory.

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this.

In [ ]:
!lerobot-train \
  --policy.type=smolvla \
  --policy.device=cuda \
  --policy.repo_id=YOUR_HF_USERNAME/YOUR_MODEL_NAME \
  --dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET \
  --batch_size=64 \
  --steps=20000 \
  --output_dir=outputs/train/my_smolvla \
  --job_name=my_smolvla_training \
  --wandb.enable=false

Sometimes after training, you may notice that the model underperforms and cannot solve the task properly. Sometimes this is due to poor data quality, but sometimes the model simply needs more training. To continue training from a previously trained model, use `--policy.pretrained_path=username/path_to_model` and paste the path to the model you trained previously here.

In [16]:
ls /content/drive/colab/MyDrive/lerobot_ds/pickup_blue_6

data/  images/  meta/  videos/


In [10]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Train pi0

In [5]:
!lerobot-train \
  --policy.type=pi0 \
  --policy.device=cuda \
  --policy.repo_id=lerobot/moonbot \
  --dataset.repo_id=gdiazsrl/moonbot_pickup_block_pickup_blue_6 \
  --dataset.root=/content/drive/colab/MyDrive/lerobot_ds/pickup_blue_6 \
  --policy.pretrained_path=lerobot/pi0_base \
  --policy.compile_model=false \
  --policy.gradient_checkpointing=true \
  --policy.dtype=bfloat16 \
  --policy.freeze_vision_encoder=false \
  --policy.train_expert_only=false \
  --batch_size=16 \
  --steps=3000 \
  --output_dir=/content/drive/colab/MyDrive/lerobot_ds/train/pickupblue6_pi0_colab \
  --job_name=pi0_training_colab \
  --num_workers=2 \
  --wandb.enable=true \
  --policy.push_to_hub=false \
  --save_freq=500 \
  --config_path=/content/drive/colab/MyDrive/lerobot_ds/train/pickupblue6_pi0_colab/checkpoints/last/pretrained_model/train_config.json \
  --resume=true

/bin/bash: line 1: lerobot-train: command not found


In [4]:
from google.colab import userdata
userdata.get('WANDB_API_KEY')
import os
import wandb
from google.colab import userdata

# Safely pull the key from your Colab Secrets
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')

wandb.finish()
!wandb sync --clean

No runs older than 24 hours found
